In [21]:
# 1. Installa swig, una dipendenza necessaria per la libreria
!apt-get update
!apt-get install -y swig

# 2. Clona il repository ufficiale
!git clone https://github.com/yvan674/obb_anns.git
%cd obb_anns

# 3. Esegui lo script di setup in modalità develop
!python setup.py develop

# 4. Torna alla directory di lavoro principale del notebook
%cd ..

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRe

In [ ]:
import os
import json
from collections import Counter

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
print("📂 Caricamento dataset...")
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

print("\n" + "="*60)
print("🔍 ANALISI DELLA STRUTTURA DEL DATASET")
print("="*60)

# 1. Tipo e chiavi principali
print(f"\n1️⃣ Tipo di train_data: {type(train_data)}")
print(f"   Chiavi principali: {list(train_data.keys())}")

# 2. Analisi di 'categories'
categories = train_data.get('categories', {})
print(f"\n2️⃣ Categorie:")
print(f"   Tipo: {type(categories)}")
print(f"   Numero di categorie: {len(categories)}")

# Mostra le prime 3 categorie
if categories:
    print("\n   Prime 3 categorie:")
    for i, (cat_id, cat_info) in enumerate(list(categories.items())[:3]):
        print(f"     {cat_id}: {cat_info}")

# 3. Analisi di 'images'
images = train_data.get('images', [])
print(f"\n3️⃣ Immagini:")
print(f"   Tipo: {type(images)}")
print(f"   Numero di immagini: {len(images)}")

if images:
    print("\n   Prima immagine:")
    first_img = images[0]
    for key, value in first_img.items():
        print(f"     {key}: {type(value)} - {value if not isinstance(value, list) else f'lista con {len(value)} elementi'}")

# 4. Analisi di 'annotations'
annotations = train_data.get('annotations', {})
print(f"\n4️⃣ Annotazioni:")
print(f"   Tipo: {type(annotations)}")
print(f"   Numero di annotazioni: {len(annotations)}")

if annotations:
    # Prendi la prima annotazione
    first_ann_id = list(annotations.keys())[0]
    first_ann = annotations[first_ann_id]
    print(f"\n   Prima annotazione (ID: {first_ann_id}):")
    for key, value in first_ann.items():
        value_type = type(value)
        value_preview = value if not isinstance(value, list) else f"lista con {len(value)} elementi: {value[:3] if len(value) > 3 else value}"
        print(f"     {key}: {value_type} - {value_preview}")

# 5. Verifica specifica del problema: category_id
print(f"\n5️⃣ Analisi specifica di 'category_id' nelle annotazioni:")
category_id_types = []
category_id_values = []

for ann_id, ann in list(annotations.items())[:100]:  # Controlla prime 100
    cat_id = ann.get('category_id', ann.get('cat_id'))
    category_id_types.append(type(cat_id).__name__)
    category_id_values.append(cat_id)

type_counter = Counter(category_id_types)
print(f"   Tipi di category_id trovati: {dict(type_counter)}")

# Mostra esempi di valori problematici
print("\n   Esempi di valori category_id:")
for i, val in enumerate(category_id_values[:5]):
    print(f"     Esempio {i}: {val} (tipo: {type(val).__name__})")

# Se category_id è una lista, mostriamo la struttura
if any(isinstance(v, list) for v in category_id_values):
    print("\n   ⚠️ ATTENZIONE: category_id è una lista in alcune annotazioni!")
    print("   Esempio di lista category_id:")
    for v in category_id_values:
        if isinstance(v, list):
            print(f"     {v} (lunghezza: {len(v)})")
            break

# 6. Verifica struttura delle bounding box
print(f"\n6️⃣ Analisi bounding box:")
bbox_types = []
bbox_formats = []

for ann_id, ann in list(annotations.items())[:100]:
    for bbox_key in ['a_bbox', 'bbox']:
        if bbox_key in ann:
            bbox = ann[bbox_key]
            bbox_types.append(type(bbox).__name__)
            if isinstance(bbox, list):
                bbox_formats.append(f"{len(bbox)} elementi")
            break

print(f"   Tipi bbox: {Counter(bbox_types)}")
if bbox_formats:
    print(f"   Formati bbox: {Counter(bbox_formats)}")

# 7. Esempio di annotazione completa
print("\n7️⃣ Esempio di annotazione completa (la prima trovata):")
if annotations:
    sample_ann_id = list(annotations.keys())[0]
    sample_ann = annotations[sample_ann_id]
    print(json.dumps(sample_ann, indent=2)[:500] + "..." if len(json.dumps(sample_ann)) > 500 else json.dumps(sample_ann, indent=2))

print("\n" + "="*60)
print("✅ Analisi completata")

In [ ]:
import os
import json
import shutil
from tqdm import tqdm
from collections import Counter

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
OUTPUT_DIR = '/kaggle/working/deepscores_yolo'

# Crea directory di output
os.makedirs(os.path.join(OUTPUT_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'labels'), exist_ok=True)

# Carica i dati
print("📂 Caricamento dataset...")
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

with open(os.path.join(BASE_DIR, "deepscores_test.json"), 'r') as f:
    test_data = json.load(f)

# 1. Crea mapping delle categorie (da ID stringa a indice progressivo)
categories = train_data.get('categories', {})
cat_mapping = {}
# Ordina per ID per consistenza
for idx, (cat_id, cat_info) in enumerate(sorted(categories.items(), key=lambda x: int(x[0]))):
    cat_mapping[cat_id] = idx  # cat_id è stringa come '135', '208'

print(f"✅ Categorie mappate: {len(cat_mapping)} classi (0..{len(cat_mapping)-1})")
print(f"   Esempio mapping: '135' -> {cat_mapping.get('135', 'N/A')}, '208' -> {cat_mapping.get('208', 'N/A')}")

# 2. Crea dizionari per accesso veloce
train_annotations = train_data.get('annotations', {})
test_annotations = test_data.get('annotations', {})

# Crea dizionario immagini per ID (converti img_id in int per consistenza)
train_images_by_id = {img['id']: img for img in train_data.get('images', [])}
test_images_by_id = {img['id']: img for img in test_data.get('images', [])}

print(f"✅ Annotazioni train: {len(train_annotations)}")
print(f"✅ Annotazioni test: {len(test_annotations)}")

# 3. Funzione per convertire bbox in formato YOLO
def convert_bbox_to_yolo(bbox, img_width, img_height):
    """
    Converte bbox in formato [x, y, width, height] (COCO)
    in formato YOLO [x_center, y_center, width, height] (normalizzati)
    """
    x, y, w, h = bbox
    x_center = (x + w/2) / img_width
    y_center = (y + h/2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    # Assicura che i valori siano entro [0,1]
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    w_norm = max(0.0, min(1.0, w_norm))
    h_norm = max(0.0, min(1.0, h_norm))
    return [x_center, y_center, w_norm, h_norm]

# 4. Funzione per processare un dataset
def process_dataset(images_by_id, annotations_dict, name):
    print(f"\n📦 Processando {name.upper()}...")
    
    # Raggruppa annotazioni per img_id (converti img_id in int)
    anns_by_img = {}
    for ann_id, ann in annotations_dict.items():
        img_id = int(ann['img_id'])  # img_id è stringa, converti a int
        if img_id not in anns_by_img:
            anns_by_img[img_id] = []
        anns_by_img[img_id].append(ann)
    
    print(f"  Immagini con annotazioni: {len(anns_by_img)}")
    
    images_processed = 0
    total_annotations = 0
    categories_used = Counter()
    
    # Per ogni immagine, crea il file .txt
    for img_id, annotations_list in tqdm(anns_by_img.items(), desc=f"  {name}"):
        img = images_by_id.get(img_id)
        if img is None:
            print(f"\n  ⚠️ Immagine {img_id} non trovata nel dizionario")
            continue
        
        img_width = img['width']
        img_height = img['height']
        img_filename = img['filename']
        
        # Percorso del file di label
        label_filename = os.path.splitext(img_filename)[0] + '.txt'
        label_path = os.path.join(OUTPUT_DIR, 'labels', label_filename)
        
        # Scrivi le annotazioni (una riga per ogni categoria)
        with open(label_path, 'w') as f:
            for ann in annotations_list:
                # Prende la bounding box allineata (a_bbox)
                bbox = ann.get('a_bbox')
                if bbox is None:
                    continue
                
                # cat_id è una lista di stringhe, es. ['135', '208']
                cat_ids = ann.get('cat_id', [])
                if not isinstance(cat_ids, list):
                    cat_ids = [cat_ids]  # se per caso non è lista, converti
                
                # Per ogni categoria, crea una riga separata
                for cat_id in cat_ids:
                    cat_id_str = str(cat_id)
                    if cat_id_str not in cat_mapping:
                        continue
                    
                    yolo_class_id = cat_mapping[cat_id_str]
                    yolo_bbox = convert_bbox_to_yolo(bbox, img_width, img_height)
                    
                    # Scrive nel formato: class x_center y_center width height
                    f.write(f"{yolo_class_id} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")
                    total_annotations += 1
                    categories_used[yolo_class_id] += 1
        
        # Copia l'immagine
        src_img_path = os.path.join(BASE_DIR, 'images', img_filename)
        dst_img_path = os.path.join(OUTPUT_DIR, 'images', img_filename)
        
        if os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)
            images_processed += 1
    
    print(f"  ✅ {name}: {images_processed} immagini processate")
    print(f"     Totale righe annotazioni: {total_annotations}")
    print(f"     Categorie utilizzate: {len(categories_used)}")
    
    return categories_used

# 5. Esegui la conversione
print("\n" + "="*60)
print("🚀 INIZIO CONVERSIONE")
print("="*60)

train_cats = process_dataset(train_images_by_id, train_annotations, 'train')
test_cats = process_dataset(test_images_by_id, test_annotations, 'test')

# 6. Riepilogo finale
print("\n" + "="*60)
print("📊 RIEPILOGO CONVERSIONE")
print("="*60)

images_dir = os.path.join(OUTPUT_DIR, 'images')
labels_dir = os.path.join(OUTPUT_DIR, 'labels')

print(f"  Immagini totali: {len(os.listdir(images_dir))}")
print(f"  Label file totali: {len(os.listdir(labels_dir))}")

# 7. Crea file YAML per YOLO
# Crea lista nomi delle categorie in ordine di indice
category_names = {}
for cat_id, cat_info in sorted(categories.items(), key=lambda x: int(x[0])):
    idx = cat_mapping[cat_id]
    category_names[idx] = cat_info.get('name', f'class_{cat_id}')

# Crea YAML
yaml_content = f"""# DeepScores Dense Dataset (con duplicazione annotazioni multi-categoria)
# Ogni annotazione originale con 2 categorie genera 2 righe nel file .txt

path: {OUTPUT_DIR}
train: images
val: images

nc: {len(cat_mapping)}
names: [{', '.join([f"'{category_names[i]}'" for i in range(len(cat_mapping))])}]
"""

yaml_path = '/kaggle/working/deepscores_dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ File YAML creato: {yaml_path}")

# 8. Verifica rapida
print("\n🔍 Verifica rapida (prime 3 righe di un label file):")
sample_labels = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
if sample_labels:
    sample_path = os.path.join(labels_dir, sample_labels[0])
    print(f"  File: {sample_labels[0]}")
    with open(sample_path, 'r') as f:
        lines = f.readlines()[:3]
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, xc, yc, w, h = parts
                print(f"    classe {class_id} ({category_names.get(int(class_id), '?')}) -> centro=({xc},{yc}) size=({w},{h})")

### CHECK BOX

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# Percorsi versione dense
BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

# Carica JSON
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Prendi prima immagine
img_info = data['images'][0]
img_file = img_info['filename']
img_w = img_info['width']
img_h = img_info['height']

# Prendi PRIMA annotazione
first_ann_id = img_info['ann_ids'][0]
ann = data['annotations'][first_ann_id]

print(f"📸 Immagine: {img_file}")
print(f"📏 Dimensioni: {img_w} x {img_h}")
print(f"\n📦 PRIMA ANNOTAZIONE (ID={first_ann_id}):")
print(f"   {ann}")

# Carica immagine
img = cv2.imread(f'{IMG_PATH}/{img_file}')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Mostra immagine
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)

# Prende bbox
bbox = ann.get('a_bbox') or ann.get('o_bbox')
if bbox:
    x, y, w, h = bbox
    print(f"\n📦 Bounding box: x={x}, y={y}, w={w}, h={h}")
    
    # Disegna solo questa bbox
    rect = plt.Rectangle((x, y), w, h, linewidth=3, edgecolor='red', facecolor='none')
    plt.gca().add_patch(rect)
    
    # Testo classe
    cat_id = ann.get('cat_id', '?')
    if isinstance(cat_id, list):
        cat_id = cat_id[0]
    plt.text(x, y-10, f'Class {cat_id}', color='red', fontsize=12, weight='bold')
    
    # Zoom intorno alla bbox
    margin = 0
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(img_w, x + w + margin)
    y2 = min(img_h, y + h + margin)
    plt.xlim(x1, x2)
    plt.ylim(y2, y1)

plt.title(f'{img_file} - SOLO prima annotazione')
plt.axis('off')
plt.show()

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

with open(JSON_FILE, 'r') as f:
    data = json.load(f)

def verifica_bbox(ann, img_w, img_h):
    """Verifica se una bbox è nel formato corretto"""
    bbox = ann.get('a_bbox')
    if not bbox:
        return "NO_BBOX"
    
    if len(bbox) != 4:
        return f"LUNGHEZZA_ERRATA:{len(bbox)}"
    
    x, y, w, h = bbox
    
    # Controlla tipo
    if isinstance(x, float) and x < 1 and isinstance(w, float) and w < 1:
        return "NORMALIZZATA"  # Probabilmente già in [0,1]
    
    if isinstance(x, (int, float)) and isinstance(w, (int, float)):
        # Controlla se è [x,y,w,h] o [x1,y1,x2,y2]
        if w > 10 and x + w < img_w + 10:  # formato [x,y,w,h]
            return "FORMATO_CORRETTO"
        elif w > 10 and w > img_w:  # potrebbe essere [x1,y1,x2,y2]
            return "FORSE_X1Y1X2Y2"
    
    return "FORMATO_INDEFINITO"

print("="*60)
print("🔍 VERIFICA FORMATO BB0X SU 20 ANNOTAZIONI CASUALI")
print("="*60)

test_images = data['images'][:5]
problemi = []

for img_info in test_images:
    img_w = img_info['width']
    img_h = img_info['height']
    img_file = img_info['filename']
    
    print(f"\n📸 {img_file} ({img_w}x{img_h})")
    
    for ann_id in img_info['ann_ids'][:3]:  # prime 3 per immagine
        ann = data['annotations'][ann_id]
        risultato = verifica_bbox(ann, img_w, img_h)
        
        bbox = ann.get('a_bbox', 'N/A')
        print(f"   ID={ann_id}: {risultato} → {bbox}")
        
        if risultato != "FORMATO_CORRETTO":
            problemi.append((img_file, ann_id, risultato, bbox))

print("\n" + "="*60)
print("📊 RIEPILOGO PROBLEMI")
print("="*60)

if problemi:
    print(f"⚠️ Trovati {len(problemi)} potenziali problemi:")
    for p in problemi[:10]:
        print(f"   {p[0]} - anno{p[1]}: {p[2]} → {p[3]}")
else:
    print("✅ Nessun problema rilevato - formato [x,y,w,h] confermato!")

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

# Prendi un'immagine e le sue annotazioni
images = train_data.get('images', [])
sample_img = images[0]  # Prima immagine
img_id = sample_img['id']

print(f"📸 Immagine: {sample_img['filename']}")
print(f"   Dimensioni dichiarate: {sample_img['width']} x {sample_img['height']}")
print()

# Carica l'immagine reale per vedere le dimensioni effettive
img_path = os.path.join(BASE_DIR, 'images', sample_img['filename'])
img_real = cv2.imread(img_path)
if img_real is not None:
    h, w = img_real.shape[:2]
    print(f"   Dimensioni reali immagine: {w} x {h}")
    print(f"   Dimensioni dichiarate vs reali: {sample_img['width']==w}, {sample_img['height']==h}")
print()

# Trova le annotazioni per questa immagine
img_annotations = []
for ann_id, ann in train_data['annotations'].items():
    if int(ann['img_id']) == img_id:
        img_annotations.append(ann)

print(f"Numero annotazioni: {len(img_annotations)}")
print()

# Analizza le prime 5 annotazioni
print("="*70)
print("ANALISI PRIME 5 ANNOTAZIONI")
print("="*70)

for i, ann in enumerate(img_annotations[:5]):
    bbox = ann.get('a_bbox')
    if bbox:
        x1, y1, x2, y2 = bbox
        print(f"\nAnnotazione {i+1}:")
        print(f"  a_bbox: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
        
        # Calcola larghezza e altezza
        w = abs(x2 - x1)
        h = abs(y2 - y1)
        print(f"  Larghezza calcolata: {w:.1f}")
        print(f"  Altezza calcolata: {h:.1f}")
        print(f"  Area: {w*h:.1f}")
        
        # Verifica se le coordinate sono dentro l'immagine
        img_w, img_h = sample_img['width'], sample_img['height']
        print(f"  Dentro immagine ({img_w}x{img_h}): x1={x1>=0}, y1={y1>=0}, x2={x2<=img_w}, y2={y2<=img_h}")

# Analisi statistica
print("\n" + "="*70)
print("ANALISI STATISTICA SU 100 ANNOTAZIONI")
print("="*70)

x1_vals, y1_vals, x2_vals, y2_vals = [], [], [], []
for ann in img_annotations[:100]:
    bbox = ann.get('a_bbox')
    if bbox:
        x1, y1, x2, y2 = bbox
        x1_vals.append(x1)
        y1_vals.append(y1)
        x2_vals.append(x2)
        y2_vals.append(y2)

print(f"x1: min={min(x1_vals):.1f}, max={max(x1_vals):.1f}, media={np.mean(x1_vals):.1f}")
print(f"y1: min={min(y1_vals):.1f}, max={max(y1_vals):.1f}, media={np.mean(y1_vals):.1f}")
print(f"x2: min={min(x2_vals):.1f}, max={max(x2_vals):.1f}, media={np.mean(x2_vals):.1f}")
print(f"y2: min={min(y2_vals):.1f}, max={max(y2_vals):.1f}, media={np.mean(y2_vals):.1f}")

# Visualizza una singola annotazione per capire
print("\n" + "="*70)
print("VISUALIZZAZIONE SINGOLA ANNOTAZIONE")
print("="*70)

# Seleziona una annotazione
ann = img_annotations[0]
bbox = ann.get('a_bbox')
x1, y1, x2, y2 = bbox

# Carica l'immagine
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Immagine completa
axes[0].imshow(img)
# Disegna rettangolo
rect = plt.Rectangle((min(x1,x2), min(y1,y2)), abs(x2-x1), abs(y2-y1), 
                     linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title(f'Bounding box: [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]')
axes[0].axis('off')

# Zoom sull'area della bounding box
x_center = (x1 + x2) / 2
y_center = (y1 + y2) / 2
zoom_w = abs(x2 - x1) * 1.5
zoom_h = abs(y2 - y1) * 1.5

x_min = max(0, x_center - zoom_w/2)
x_max = min(img.shape[1], x_center + zoom_w/2)
y_min = max(0, y_center - zoom_h/2)
y_max = min(img.shape[0], y_center + zoom_h/2)

img_zoom = img[int(y_min):int(y_max), int(x_min):int(x_max)]
axes[1].imshow(img_zoom)
# Ricalcola rettangolo nello zoom
rect_zoom = plt.Rectangle((max(0, x1 - x_min), max(0, y1 - y_min)), 
                          abs(x2-x1), abs(y2-y1), 
                          linewidth=2, edgecolor='red', facecolor='none')
axes[1].add_patch(rect_zoom)
axes[1].set_title(f'Zoom sulla bounding box\nCategoria: {ann.get("cat_id")}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Stampa info categoria
cat_ids = ann.get('cat_id', [])
print(f"\nCategorie: {cat_ids}")
for c in cat_ids:
    if c is not None:
        cat_id_int = int(c)
        cat_name = train_data['categories'].get(str(cat_id_int), {}).get('name', 'unknown')
        print(f"  - ID {cat_id_int}: {cat_name}")

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
import random
import numpy as np


# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

# Crea mapping delle categorie
categories = train_data.get('categories', {})
cat_id_to_name = {int(cat_id): cat_info['name'] for cat_id, cat_info in categories.items()}

def visualize_bbox(image_id, annotations_dict, images_list, max_annotations=100):
    """
    Visualizza le bounding box usando il formato corretto [x1, y1, x2, y2]
    """
    # Trova l'immagine
    img_info = None
    for img in images_list:
        if img['id'] == image_id:
            img_info = img
            break
    
    if img_info is None:
        print(f"Immagine con ID {image_id} non trovata")
        return None
    
    # Carica l'immagine
    img_path = os.path.join(BASE_DIR, 'images', img_info['filename'])
    img = cv2.imread(img_path)
    if img is None:
        print(f"Impossibile caricare l'immagine: {img_path}")
        return None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Trova le annotazioni per questa immagine
    img_annotations = []
    for ann_id, ann in annotations_dict.items():
        if int(ann['img_id']) == image_id:
            img_annotations.append(ann)
    
    print(f"\n📸 {img_info['filename']}")
    print(f"   Dimensioni: {img_info['width']} x {img_info['height']}")
    print(f"   Annotazioni totali: {len(img_annotations)}")
    print(f"   Visualizzate: {min(max_annotations, len(img_annotations))}")
    
    # Crea figura
    fig, ax = plt.subplots(1, 1, figsize=(15, 12))
    ax.imshow(img)
    
    # Colori per categorie
    random.seed(422)
    colors = {}
    
    for ann in img_annotations[:max_annotations]:
        bbox = ann.get('a_bbox')
        if bbox is None or len(bbox) != 4:
            continue
        
        x1, y1, x2, y2 = bbox
        
        # Assicura coordinate corrette
        x_min, x_max = min(x1, x2), max(x1, x2)
        y_min, y_max = min(y1, y2), max(y1, y2)
        w = x_max - x_min
        h = y_max - y_min
        
        # Ottieni categoria
        cat_ids = ann.get('cat_id', [])
        if cat_ids and cat_ids[0] is not None:
            cat_id = int(cat_ids[0])
            if cat_id not in colors:
                colors[cat_id] = (random.random(), random.random(), random.random())
            color = colors[cat_id]
            
            # Disegna rettangolo
            rect = plt.Rectangle((x_min, y_min), w, h, 
                                linewidth=1.5, 
                                edgecolor=color, 
                                facecolor='none')
            ax.add_patch(rect)
            
            # Etichetta
            cat_name = cat_id_to_name.get(cat_id, f'c{cat_id}')
            ax.text(x_min, y_min-3, cat_name, fontsize=7, color=color,
                   bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.7))
    
    ax.set_title(f'{img_info["filename"]} - {len(img_annotations)} annotazioni totali')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return fig

# Test con alcune immagini
images = train_data.get('images', [])

# Mostra prime 3 immagini
for i in range(min(3, len(images))):
    print(f"\n{'='*60}")
    print(f"IMMAGINE {i+1}")
    print('='*60)
    visualize_bbox(images[i]['id'], train_data['annotations'], images)

In [ ]:
import os
import yaml

# Percorsi
OUTPUT_DIR = '/kaggle/working/deepscores_yolo'
YAML_PATH = '/kaggle/working/deepscores_dataset.yaml'

# Verifica che la directory esista
if not os.path.exists(OUTPUT_DIR):
    print(f"❌ Directory non trovata: {OUTPUT_DIR}")
    print("   Devi prima eseguire lo script di conversione!")
else:
    print(f"✅ Directory trovata: {OUTPUT_DIR}")
    
    # Conta immagini e label
    images_dir = os.path.join(OUTPUT_DIR, 'images')
    labels_dir = os.path.join(OUTPUT_DIR, 'labels')
    
    num_images = len(os.listdir(images_dir)) if os.path.exists(images_dir) else 0
    num_labels = len(os.listdir(labels_dir)) if os.path.exists(labels_dir) else 0
    
    print(f"   Immagini: {num_images}")
    print(f"   Label: {num_labels}")
    
    # Carica il dataset originale per ottenere il numero di classi
    BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
    with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
        train_data = json.load(f)
    
    categories = train_data.get('categories', {})
    num_classes = len(categories)
    
    # Crea lista nomi classi (ordine per ID)
    class_names = []
    for i in range(num_classes):
        class_names.append(f"class_{i}")
    
    # Crea YAML
    yaml_content = {
        'path': OUTPUT_DIR,
        'train': 'images',
        'val': 'images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Salva YAML
    with open(YAML_PATH, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"\n✅ File YAML creato: {YAML_PATH}")
    print("\nContenuto:")
    print(f"  path: {OUTPUT_DIR}")
    print(f"  train: images")
    print(f"  val: images")
    print(f"  nc: {num_classes}")
    print(f"  names: [{num_classes} classi]")

In [ ]:
from ultralytics import YOLO
import optuna

def objective(trial):
    # Suggerisci iperparametri (in modo intelligente)
    lr0 = trial.suggest_float("lr0", 0.0005, 0.002, log=True)
    box = trial.suggest_float("box", 6.0, 9.0)
    cls = trial.suggest_float("cls", 0.8, 1.6)
    
    # Addestra modello
    model = YOLO("yolo11n.pt")
    results = model.train(
        data='/kaggle/working/deepscores_dataset.yaml',
        epochs=30,
        optimizer='AdamW',
        batch=6,
        workers=2,
        imgsz=320,
        lr0=lr0,
        box=box,
        cls=cls,
        device=0,
        verbose=False,
    )
    
    # Restituisci metrica da massimizzare
    return results.maps[0]  # mAP50-95

# Crea studio Optuna
study = optuna.create_study(
    direction="maximize",  # vogliamo mAP più alto
    pruner=optuna.pruners.MedianPruner()  # ferma trial inutili
)

# Esegui 50 trial (meno di grid search equivalente!)
study.optimize(objective, n_trials=20)

print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP: {study.best_value:.4f}")

## TEST POST ACCORGIMENTI IVAN

In [32]:
from ultralytics import YOLO
import optuna
import torch

def objective(trial):
    # Suggerisci iperparametri con range più ampi
    lr0 = trial.suggest_float("lr0", 0.0006, 0.001, log=True)  # LR iniziale più basso
    lrf = trial.suggest_float("lrf", 0.005, 0.02, log=True)    # LR finale = lr0 * lrf
    box = trial.suggest_float("box", 6.0, 9.0)
    cls = trial.suggest_float("cls", 0.8, 1.6)
    momentum = trial.suggest_float("momentum", 0.85, 0.95)      # Aggiunto momentum
    warmup_epochs = trial.suggest_int("warmup_epochs", 3, 10)   # Warmup epoche
    
    # Usa yolo11m (medium) invece di nano
    model = YOLO("yolo11m.pt")
    
    results = model.train(
        data='/kaggle/working/deepscores_dataset.yaml',
        epochs=100,                    
        patience=30,                    # Early stopping più paziente
        optimizer='AdamW',
        batch=4,                        # Batch più piccolo per yolo11m
        workers=2,
        imgsz=416,                      # Aumentato per dettaglio (da 320 a 416)
        lr0=lr0,
        lrf=lrf,                        # LR finale
        momentum=momentum,
        weight_decay=0.0005,            # Default, lasciato invariato
        warmup_epochs=warmup_epochs,    # Warmup configurabile
        warmup_bias_lr=0.1,
        warmup_momentum=0.8,
        box=box,
        cls=cls,
        device=0,
        verbose=False,
        cache=False,                    # IMPORTANTE: evita di usare RAM come cache
        rect=True,
        amp=True,                       # Mixed precision per risparmiare VRAM
        exist_ok=True,
    )
    
    # Restituisci mAP50-95
    return results.maps[0]

# Monitoraggio VRAM
print("=" * 60)
print("🔍 MONITORAGGIO RISORSE")
print("=" * 60)


# Crea studio Optuna
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=3,
        n_warmup_steps=10,      # Aspetta 10 epoche prima di potare
        interval_steps=5
    )
)

print("\n🚀 Avvio tuning YOLO11m con 150 epoche...")
print("⚠️ Attenzione: Ogni trial richiederà diverse ore")
print("=" * 60)

# Riduci trial a 5-8 per evitare timeout
study.optimize(objective, n_trials=5, timeout=28800)  # Timeout 8 ore

print("\n" + "=" * 60)
print("📊 RISULTATI FINALI")
print("=" * 60)
print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP50-95: {study.best_value:.4f}")
print(f"Miglior mAP50-95 percentuale: {study.best_value*100:.2f}%")

# Salva risultati
import json
with open('/kaggle/working/best_params_optuna.json', 'w') as f:
    json.dump({
        'best_params': study.best_params,
        'best_value': study.best_value,
        'best_trial': study.best_trial.number
    }, f, indent=2)
print("✅ Parametri salvati in /kaggle/working/best_params_optuna.json")

# Mostra tutti i trial
print("\n📋 Storico trial:")
df = study.trials_dataframe()
for i, row in df.iterrows():
    if row['state'] == 'COMPLETE':
        print(f"  Trial {int(row['number'])}: mAP={row['value']:.4f} | "
              f"lr0={row['params_lr0']:.5f} | lrf={row['params_lrf']:.4f} | "
              f"box={row['params_box']:.2f} | cls={row['params_cls']:.2f}")

[I 2026-06-04 16:00:59,776] A new study created in memory with name: no-name-91044036-9ceb-48de-bae4-fe8d28924d2b
[W 2026-06-04 16:00:59,782] Trial 0 failed with parameters: {'lr0': 0.0006612773855808985, 'lrf': 0.007157205351775995, 'box': 8.781422036521697, 'cls': 1.4695866234724204, 'momentum': 0.905231772091038, 'warmup_epochs': 3} because of the following error: AttributeError("module 'torch' has no attribute '_utils'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_58/597471130.py", line 15, in objective
    model = YOLO("yolo11m.pt")
            ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 78, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py"

🔍 MONITORAGGIO RISORSE

🚀 Avvio tuning YOLO11m con 150 epoche...
⚠️ Attenzione: Ogni trial richiederà diverse ore


AttributeError: module 'torch' has no attribute '_utils'

# TEST con DeepScoresV2

In [ ]:
from ultralytics import YOLO
import optuna
import os
import json

DATASET_PATH = '/kaggle/input/datasets/joshuastiller/deepscoresv2/ds2_complete'

print(f"✅ Dataset trovato: {DATASET_PATH}")

# Verifica tutti i file JSON disponibili
print("\n📋 File JSON disponibili:")
train_files = []
test_files = []

for i in range(13):  # 0-12
    train_file = f'{DATASET_PATH}/deepscores-complete-{i}_train.json'
    test_file = f'{DATASET_PATH}/deepscores-complete-{i}_test.json'
    
    if os.path.exists(train_file):
        train_files.append(train_file)
        print(f"  ✅ train-{i}.json")
    if os.path.exists(test_file):
        test_files.append(test_file)
        print(f"  ✅ test-{i}.json")

print(f"\nTotale: {len(train_files)} file train, {len(test_files)} file test")

# Ispeziona la struttura del primo JSON
print("\n🔍 Ispezione struttura primo JSON:")
with open(train_files[0], 'r') as f:
    sample = json.load(f)
    print(f"Chiavi principali: {sample.keys()}")
    if 'annotations' in sample and len(sample['annotations']) > 0:
        print(f"Tipo annotations[0]: {type(sample['annotations'][0])}")
        print(f"Contenuto annotations[0]: {sample['annotations'][0][:200] if isinstance(sample['annotations'][0], str) else sample['annotations'][0]}")
    if 'images' in sample and len(sample['images']) > 0:
        print(f"Tipo images[0]: {type(sample['images'][0])}")
        print(f"Contenuto images[0]: {sample['images'][0]}")

def merge_json_files(json_files, output_path):
    merged = {
        'images': [],
        'annotations': [],
        'categories': None
    }
    
    img_id_offset = 0
    ann_id_offset = 0
    
    for jf in json_files:
        with open(jf, 'r') as f:
            data = json.load(f)
        
        # Copia categorie (sono uguali per tutti)
        if merged['categories'] is None:
            merged['categories'] = data['categories']
        
        # Aggiungi immagini
        for img in data['images']:
            img_copy = img.copy() if hasattr(img, 'copy') else img
            if isinstance(img_copy, dict):
                img_copy['id'] = img_copy.get('id', 0) + img_id_offset
            merged['images'].append(img_copy)
        
        # Aggiungi annotazioni (gestisci sia dict che str)
        for ann in data['annotations']:
            if isinstance(ann, str):
                # Se è stringa, prova a parsare come JSON
                try:
                    ann_copy = json.loads(ann)
                except:
                    print(f"⚠️ Warning: annotazione non parsabile: {ann[:100]}")
                    continue
            else:
                ann_copy = ann.copy() if hasattr(ann, 'copy') else ann
            
            if isinstance(ann_copy, dict):
                ann_copy['id'] = ann_copy.get('id', 0) + ann_id_offset
                ann_copy['img_id'] = ann_copy.get('img_id', 0) + img_id_offset
            merged['annotations'].append(ann_copy)
        
        # Aggiorna offset (solo per dict validi)
        valid_images = [img for img in merged['images'] if isinstance(img, dict) and 'id' in img]
        if valid_images:
            img_id_offset = max([img['id'] for img in valid_images]) + 1
        
        valid_anns = [ann for ann in merged['annotations'] if isinstance(ann, dict) and 'id' in ann]
        if valid_anns:
            ann_id_offset = max([ann['id'] for ann in valid_anns]) + 1
    
    # Salva JSON unito
    with open(output_path, 'w') as f:
        json.dump(merged, f)
    
    print(f"✅ Creato {output_path}: {len(merged['images'])} immagini, {len(merged['annotations'])} annotazioni")
    return output_path

# Unisci train e test separatamente
print("\n🔄 Unione file train...")
train_json = merge_json_files(train_files, '/kaggle/working/deepscores_train_merged.json')

print("\n🔄 Unione file test...")
test_json = merge_json_files(test_files, '/kaggle/working/deepscores_test_merged.json')

# Carica il train unito per ottenere il numero di categorie
with open(train_json, 'r') as f:
    train_data = json.load(f)
    num_classes = len(train_data.get('categories', []))

# Crea file data.yaml per YOLO
data_yaml = f"""
path: /kaggle/working
train: deepscores_train_merged.json
val: deepscores_test_merged.json

nc: {num_classes}
names: [\"class_{i}\" for i in range({num_classes})]
"""

with open('/kaggle/working/deepscores_dataset.yaml', 'w') as f:
    f.write(data_yaml)

print(f"\n✅ Creato deepscores_dataset.yaml con {num_classes} classi")
print("✅ Dataset pronto per YOLO!")

# RIORDINIAMO!

### Script di analisi formati

In [ ]:
import json
import numpy as np
from collections import Counter

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'

with open(JSON_FILE, 'r') as f:
    data = json.load(f)

print("="*80)
print("🔍 ANALISI FORMATO BOUNDING BOX")
print("="*80)

# Statistiche
formati = []
dimensioni = []
rapporti = []
coordinate_negative = 0
bbox_zero = 0

for img_info in data['images']:
    img_w = img_info['width']
    img_h = img_info['height']
    
    for ann_id in img_info['ann_ids']:
        ann = data['annotations'][ann_id]
        bbox = ann.get('a_bbox')
        
        if not bbox or len(bbox) != 4:
            continue
        
        x1, y1, x3, y3 = bbox
        
        # Determina il formato
        if x3 > x1 and y3 > y1:
            # Può essere [x,y,w,h] o [x1,y1,x2,y2]
            w = x3 - x1
            h = y3 - y1
            
            if w <= 0 or h <= 0:
                bbox_zero += 1
                formato = "INVALIDO"
            elif w < 100 and h < 100 and w > 0 and h > 0:
                # Formato [x,y,w,h] con valori piccoli
                formato = "COCO [x,y,w,h]"
                formati.append("COCO")
            elif w > img_w * 0.8 or x3 > img_w * 0.9:
                # Formato [x1,y1,x2,y2]
                formato = "VOC [x1,y1,x2,y2]"
                formati.append("VOC")
            else:
                formato = "INDETERMINATO"
                formati.append("INDETERMINATO")
            
            dimensioni.append(w)
            if w > 0:
                rapporti.append(h/w)
        else:
            coordinate_negative += 1
            formati.append("ERRORE")

# Statistiche
counter = Counter(formati)
total = len(formati)

print(f"\n📊 STATISTICHE TOTALI:")
print(f"   Totale annotazioni analizzate: {total}")
print(f"\n   Formati rilevati:")
for formato, count in counter.most_common():
    percentuale = count/total*100
    print(f"      {formato}: {count} ({percentuale:.1f}%)")

print(f"\n   Annotazioni invalide: {bbox_zero}")
print(f"   Coordinate negative: {coordinate_negative}")

# Analisi dimensioni per formato
print(f"\n📏 ANALISI DIMENSIONI (larghezza in pixel):")
if dimensioni:
    print(f"   Media: {np.mean(dimensioni):.1f}")
    print(f"   Mediana: {np.median(dimensioni):.1f}")
    print(f"   Min: {np.min(dimensioni):.1f}")
    print(f"   Max: {np.max(dimensioni):.1f}")
    
    # Distribuzione
    piccole = sum(1 for d in dimensioni if d < 50)
    medie = sum(1 for d in dimensioni if 50 <= d < 200)
    grandi = sum(1 for d in dimensioni if d >= 200)
    
    print(f"\n   Distribuzione larghezze:")
    print(f"      Piccole (<50px): {piccole} ({piccole/total*100:.1f}%)")
    print(f"      Medie (50-200px): {medie} ({medie/total*100:.1f}%)")
    print(f"      Grandi (≥200px): {grandi} ({grandi/total*100:.1f}%)")

# Verifica coerenza
print(f"\n🎯 COERENZA:")
if counter.get("COCO [x,y,w,h]", 0) > total * 0.9:
    print("   ✅ FORMATO COERENTE: [x, y, width, height] (COCO)")
    print("   → Puoi usare la conversione standard o il formato nativo YOLO")
elif counter.get("VOC [x1,y1,x2,y2]", 0) > total * 0.9:
    print("   ✅ FORMATO COERENTE: [x1, y1, x2, y2] (Pascal VOC)")
    print("   → Devi convertire a [x, y, w, h] prima di usare YOLO")
else:
    print("   ⚠️ FORMATO MISTO: Il dataset ha formati diversi!")
    print("   → Necessaria conversione automatica che rilevi il formato")

# Mostra esempi
print(f"\n📝 ESEMPI CONCRETI:")
esempi_mostrati = 0
for img_info in data['images'][:3]:
    for ann_id in img_info['ann_ids'][:2]:
        ann = data['annotations'][ann_id]
        bbox = ann.get('a_bbox')
        if bbox and len(bbox) == 4:
            print(f"\n   Annotazione {ann_id}: {bbox}")
            print(f"   cat_id: {ann.get('cat_id')}")
            esempi_mostrati += 1
            if esempi_mostrati >= 6:
                break
    if esempi_mostrati >= 6:
        break

print("\n" + "="*80)
print("💡 CONCLUSIONE:")
if counter.get("COCO [x,y,w,h]", 0) > total * 0.95:
    print("   Usa YOLO nativamente con JSON COCO")
elif counter.get("VOC [x1,y1,x2,y2]", 0) > total * 0.95:
    print("   Converti da [x1,y1,x2,y2] a [x,y,w,h] prima di usare YOLO")
else:
    print("   Implementa conversione automatica che supporti entrambi i formati")
print("="*80)

## VOC O COCO?
DUBBIO: Il dataset DeepScoresV2Dense è MISTO alcune classi sono annotate come VOC altre come COCO? Risposta: FALSO, tutti VOC (i check successivi lo dimostrano)

### Interactive check

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

# Carica dati
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Filtra categorie valide
categories = {}
for cat_id, cat_info in data.get('categories', {}).items():
    try:
        int(cat_id)
        categories[cat_id] = cat_info
    except (ValueError, TypeError):
        continue

print("=" * 80)
print("🔍 VALUTAZIONE MANUALE: VOC vs COCO per ogni classe")
print("=" * 80)

# Raccogli annotazioni per classe
annotazioni_per_classe = defaultdict(list)

for img_info in data['images'][:200]:  # Prime 200 immagini
    img_w = img_info['width']
    img_h = img_info['height']
    
    for ann_id in img_info.get('ann_ids', []):
        ann = data['annotations'].get(ann_id)
        if ann:
            bbox = ann.get('a_bbox')
            if bbox:
                cat_ids = ann.get('cat_id', [])
                if not isinstance(cat_ids, list):
                    cat_ids = [cat_ids]
                
                for cat_id in cat_ids:
                    cat_id_str = str(cat_id)
                    if cat_id_str in categories:
                        annotazioni_per_classe[cat_id_str].append({
                            'img_file': img_info['filename'],
                            'bbox': bbox,
                            'img_w': img_w,
                            'img_h': img_h,
                            'ann_id': ann_id
                        })

# Per ogni classe, mostra un esempio come VOC e uno come COCO
classi_da_mostrare = list(annotazioni_per_classe.keys())[:30]  # Prime 30 classi

for cat_id in classi_da_mostrare:
    cat_name = categories[cat_id].get('name', f'classe_{cat_id}')
    annotazioni = annotazioni_per_classe[cat_id]
    
    if len(annotazioni) < 2:
        continue
    
    # Prendi una annotazione
    ann = annotazioni[0]
    bbox = ann['bbox']
    img_w = ann['img_w']
    img_h = ann['img_h']
    img_file = ann['img_file']
    x1, y1, x3, y3 = bbox
    
    # Calcola come VOC e COCO
    w_voc = x3 - x1
    h_voc = y3 - y1
    
    # Calcola coordinate YOLO per entrambi i formati
    # Formato VOC
    xc_voc = (x1 + w_voc/2) / img_w
    yc_voc = (y1 + h_voc/2) / img_h
    w_voc_norm = w_voc / img_w
    h_voc_norm = h_voc / img_h
    
    # Formato COCO
    xc_coco = (x1 + x3/2) / img_w
    yc_coco = (y1 + y3/2) / img_h
    w_coco_norm = x3 / img_w
    h_coco_norm = y3 / img_h
    
    # Carica immagine
    img = cv2.imread(f'{IMG_PATH}/{img_file}')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Crea figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # SINISTRA: Interpretazione VOC
    axes[0].imshow(img_rgb)
    rect_voc = plt.Rectangle((x1, y1), w_voc, h_voc, 
                             linewidth=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect_voc)
    axes[0].set_title(f'VOC: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\n'
                      f'Larghezza={w_voc:.0f}px, Altezza={h_voc:.0f}px')
    axes[0].axis('off')
    
    # DESTRA: Interpretazione COCO
    axes[1].imshow(img_rgb)
    rect_coco = plt.Rectangle((x1, y1), x3, y3, 
                              linewidth=2, edgecolor='blue', facecolor='none')
    axes[1].add_patch(rect_coco)
    axes[1].set_title(f'COCO: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\n'
                      f'Larghezza={x3:.0f}px, Altezza={y3:.0f}px')
    axes[1].axis('off')
    
    plt.suptitle(f'Classe {cat_id}: {cat_name}', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Input manuale
    print(f"\n📋 Classe {cat_id}: {cat_name}")
    print("   Quale interpretazione è CORRETTA?")
    print("    1 = VOC (rettangolo ROSSO)")
    print("    2 = COCO (rettangolo BLU)")
    print("    0 = SALTA / nessuna delle due")
    
    risposta = input("   → Scegli (0/1/2): ")
    
    # Salva risultato
    with open('/kaggle/working/valutazione_classi.txt', 'a') as f:
        f.write(f"{cat_id},{cat_name},{risposta},{w_voc},{x3}\n")
    
    plt.close()

print("\n" + "=" * 80)
print("✅ VALUTAZIONE COMPLETATA")
print("   Risultati salvati in: /kaggle/working/valutazione_classi.txt")
print("=" * 80)

# Mostra riepilogo
print("\n📊 RIEPILOGO VALUTAZIONI:")
with open('/kaggle/working/valutazione_classi.txt', 'r') as f:
    righe = f.readlines()

votazioni = {'VOC': 0, 'COCO': 0, 'SALTA': 0}
for riga in righe:
    parts = riga.strip().split(',')
    if len(parts) >= 3:
        if parts[2] == '1':
            votazioni['VOC'] += 1
        elif parts[2] == '2':
            votazioni['COCO'] += 1
        elif parts[2] == '0':
            votazioni['SALTA'] += 1

print(f"   VOC corrette: {votazioni['VOC']}")
print(f"   COCO corrette: {votazioni['COCO']}")
print(f"   Saltate: {votazioni['SALTA']}")

### Static check

In [50]:
import json
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

print("📂 Caricamento JSON...")
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Filtra categorie
categories = {}
for cat_id, cat_info in data.get('categories', {}).items():
    try:
        int(cat_id)
        categories[cat_id] = cat_info
    except (ValueError, TypeError):
        continue

print(f"✅ {len(categories)} classi trovate")

# Raccogli annotazioni
annotazioni_per_classe = defaultdict(list)
for img_info in data['images'][:200]:
    for ann_id in img_info.get('ann_ids', []):
        ann = data['annotations'].get(ann_id)
        if ann and ann.get('a_bbox'):
            cat_ids = ann.get('cat_id', [])
            if not isinstance(cat_ids, list):
                cat_ids = [cat_ids]
            for cat_id in cat_ids:
                cat_id_str = str(cat_id)
                if cat_id_str in categories:
                    annotazioni_per_classe[cat_id_str].append({
                        'img_file': img_info['filename'],
                        'bbox': ann['a_bbox'],
                        'img_w': img_info['width'],
                        'img_h': img_info['height']
                    })

# Crea PDF con tutte le classi
pdf_path = '/kaggle/working/verifica_classi_VOC_vs_COCO.pdf'
print(f"\n📄 Creazione PDF: {pdf_path}")

with PdfPages(pdf_path) as pdf:
    for cat_id in sorted(annotazioni_per_classe.keys(), key=lambda x: int(x)):
        annotazioni = annotazioni_per_classe[cat_id]
        cat_name = categories[cat_id].get('name', f'classe_{cat_id}')
        
        # Prendi una annotazione
        ann = annotazioni[0]
        bbox = ann['bbox']
        img_w = ann['img_w']
        img_h = ann['img_h']
        x1, y1, x3, y3 = bbox
        
        w_voc = x3 - x1
        h_voc = y3 - y1
        
        # Carica immagine
        img = cv2.imread(f'{IMG_PATH}/{ann["img_file"]}')
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Crea figura
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # VOC (ROSSO)
        axes[0].imshow(img_rgb)
        axes[0].add_patch(plt.Rectangle((x1, y1), w_voc, h_voc, 
                                        linewidth=2, edgecolor='red', facecolor='none'))
        axes[0].set_title(f'VOC: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\nlarghezza={w_voc:.0f}px', fontsize=10)
        axes[0].axis('off')
        
        # COCO (BLU)
        axes[1].imshow(img_rgb)
        axes[1].add_patch(plt.Rectangle((x1, y1), x3, y3, 
                                        linewidth=2, edgecolor='blue', facecolor='none'))
        axes[1].set_title(f'COCO: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\nlarghezza={x3:.0f}px', fontsize=10)
        axes[1].axis('off')
        
        fig.suptitle(f'Classe {cat_id}: {cat_name}', fontsize=12)
        plt.tight_layout()
        
        pdf.savefig(fig)
        plt.close(fig)
        
        # Progresso
        print(f"  Aggiunta classe {cat_id}: {cat_name}")

print("\n" + "=" * 60)
print(f"✅ PDF CREATO: {pdf_path}")
print("   Scarica il file e osserva ogni coppia:")
print("   - ROSSO = interpretazione VOC [x1,y1,x2,y2]")
print("   - BLU = interpretazione COCO [x,y,w,h]")
print("   - Il rettangolo CORRETTO deve circondare esattamente l'oggetto")
print("=" * 60)

# Versione alternativa: stampa tabella riassuntiva
print("\n📊 TABELLA RIASSUNTIVA (prime 50 classi):")
print("-" * 80)
print(f"{'Classe':<8} {'Nome':<30} {'w_voc':<8} {'w_coco':<8} {'Formato suggerito':<15}")
print("-" * 80)

for cat_id in sorted(annotazioni_per_classe.keys(), key=lambda x: int(x))[:50]:
    annotazioni = annotazioni_per_classe[cat_id]
    cat_name = categories[cat_id].get('name', f'classe_{cat_id}')[:28]
    ann = annotazioni[0]
    bbox = ann['bbox']
    x1, y1, x3, y3 = bbox
    w_voc = x3 - x1
    w_coco = x3
    
    img_w = ann['img_w']
    soglia_relativa = img_w * 0.10
    
    if w_voc > 0 and w_voc < soglia_relativa:
        suggerito = "VOC"
    else:
        suggerito = "COCO"
    
    print(f"{cat_id:<8} {cat_name:<30} {w_voc:<8.0f} {w_coco:<8.0f} {suggerito:<15}")

print("-" * 80)
print("\n💡 VALUTA CON I TUOI OCCHI: apri il PDF e controlla")
print("   Se il rettangolo suggerito NON circonda correttamente l'oggetto,")
print("   allora la soglia del 10% va modificata per quella classe.")

📂 Caricamento JSON...
✅ 208 classi trovate

📄 Creazione PDF: /kaggle/working/verifica_classi_VOC_vs_COCO.pdf
  Aggiunta classe 1: brace
  Aggiunta classe 2: ledgerLine
  Aggiunta classe 3: repeatDot
  Aggiunta classe 4: segno
  Aggiunta classe 5: coda
  Aggiunta classe 6: clefG
  Aggiunta classe 7: clefCAlto
  Aggiunta classe 8: clefCTenor
  Aggiunta classe 9: clefF
  Aggiunta classe 11: clef8
  Aggiunta classe 12: clef15
  Aggiunta classe 13: timeSig0
  Aggiunta classe 14: timeSig1
  Aggiunta classe 15: timeSig2
  Aggiunta classe 16: timeSig3
  Aggiunta classe 17: timeSig4
  Aggiunta classe 18: timeSig5
  Aggiunta classe 19: timeSig6
  Aggiunta classe 20: timeSig7
  Aggiunta classe 21: timeSig8
  Aggiunta classe 22: timeSig9
  Aggiunta classe 23: timeSigCommon
  Aggiunta classe 24: timeSigCutCommon
  Aggiunta classe 25: noteheadBlackOnLine
  Aggiunta classe 27: noteheadBlackInSpace
  Aggiunta classe 29: noteheadHalfOnLine
  Aggiunta classe 31: noteheadHalfInSpace
  Aggiunta classe 33:

## E' stato appurato che il formato NON è misto, ma sono tutti VOC!

## Carico il dataset

In [7]:
print("Import")

import shutil
import yaml
from tqdm import tqdm
from collections import defaultdict
import os
import json

print("DONE!")

Import
DONE!


In [8]:
#!/usr/bin/env python3
"""
DeepScoresV2 → YOLO Format Converter for Kaggle
Questo script converte il dataset DeepScoresV2 (formato JSON COCO/VOC) 
in formato YOLO (immagini + file .txt) pronto per l'addestramento.

ESEGUI QUESTA CELLA UNA SOLA VOLTA PRIMA DEL TRAINING
"""

print("=" * 70)
print("🚀 DeepScoresV2 → YOLO Format Converter")
print("=" * 70)

# ============================================
# 1. CONFIGURAZIONE
# ============================================
WORK_DIR = '/kaggle/working'
IMAGES_DIR = os.path.join(WORK_DIR, 'images')
LABELS_DIR = os.path.join(WORK_DIR, 'labels')
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

# Percorso del dataset originale su Kaggle (versione dense)
# Se usi la versione completa, cambia questo percorso
BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Crea directory
for d in [IMAGES_DIR, LABELS_DIR, DATASET_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"\n📁 Directory create:")
print(f"   Working dir: {WORK_DIR}")
print(f"   Images: {IMAGES_DIR}")
print(f"   Labels: {LABELS_DIR}")
print(f"   Dataset: {DATASET_DIR}")

# ============================================
# 2. CARICA DATASET
# ============================================
print(f"\n📂 Caricamento dataset da: {BASE_PATH}")

with open(f'{BASE_PATH}/deepscores_train.json', 'r') as f:
    train_data = json.load(f)

with open(f'{BASE_PATH}/deepscores_test.json', 'r') as f:
    test_data = json.load(f)

print(f"   Train: {len(train_data.get('images', []))} immagini")
print(f"   Test: {len(test_data.get('images', []))} immagini")

# ============================================
# 3. ANALISI CLASSI (ID originali presenti)
# ============================================
print("\n🔍 Analisi classi presenti...")

classi_presenti = set()
for ann in train_data['annotations'].values():
    cat_ids = ann.get('cat_id', [])
    if isinstance(cat_ids, list):
        for c in cat_ids:
            if c is not None:
                try:
                    classi_presenti.add(int(c))
                except (ValueError, TypeError):
                    pass
    elif cat_ids is not None:
        try:
            classi_presenti.add(int(cat_ids))
        except (ValueError, TypeError):
            pass

print(f"   Classi trovate: {len(classi_presenti)}")

# ============================================
# 4. MAPPING CLASSI (ID originale → indice YOLO)
# ============================================
print("\n🏷️  Creazione mapping classi...")

categories = train_data.get('categories', {})
id_to_idx = {}
idx_to_name = {}
name_counter = defaultdict(int)

for idx, orig_id in enumerate(sorted(classi_presenti)):
    id_to_idx[orig_id] = idx
    
    # Ottieni nome originale
    cat_info = categories.get(str(orig_id), {})
    base_name = cat_info.get('name', f'class_{orig_id}')
    
    # Crea nome unico (aggiungi suffisso per duplicati)
    name_counter[base_name] += 1
    count = name_counter[base_name]
    unique_name = base_name if count == 1 else f"{base_name}_{orig_id}"
    
    idx_to_name[idx] = unique_name

print(f"   Mapping creato: {len(idx_to_name)} classi (0..{len(idx_to_name)-1})")
print(f"   Prime 5: {list(idx_to_name.items())[:5]}")

# ============================================
# 5. FUNZIONE CONVERSIONE BBOX
# ============================================
def voc_to_yolo(bbox, img_w, img_h):
    """
    Converte bbox da formato VOC [x1, y1, x2, y2] a YOLO [x_center, y_center, w, h]
    """
    if len(bbox) != 4:
        return None
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        return None
    x_c = (x1 + w/2) / img_w
    y_c = (y1 + h/2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    # Clamping a [0,1]
    return [max(0.0, min(1.0, x_c)),
            max(0.0, min(1.0, y_c)),
            max(0.0, min(1.0, w_norm)),
            max(0.0, min(1.0, h_norm))]

# ============================================
# 6. INDICIZZAZIONE ANNOTAZIONI
# ============================================
print("\n📦 Indicizzazione annotazioni...")

anns_by_img = defaultdict(list)
for ann_id, ann in train_data['annotations'].items():
    img_id = ann.get('img_id')
    if img_id is not None:
        try:
            anns_by_img[int(img_id)].append(ann)
        except (ValueError, TypeError):
            continue

print(f"   Immagini con annotazioni: {len(anns_by_img)}")

# ============================================
# 7. CONVERSIONE E SALVATAGGIO
# ============================================
print("\n🔄 Conversione in formato YOLO...")

images_processed = 0
total_annotations = 0
errori = 0

for img in tqdm(train_data['images'], desc="   Progresso"):
    img_id = img['id']
    img_filename = img['filename']
    img_w = img['width']
    img_h = img['height']
    
    anns = anns_by_img.get(img_id, [])
    if not anns:
        continue
    
    # Crea file label .txt
    label_name = os.path.splitext(img_filename)[0] + '.txt'
    label_path = os.path.join(LABELS_DIR, label_name)
    
    with open(label_path, 'w') as f:
        for ann in anns:
            bbox = ann.get('a_bbox')
            if not bbox or len(bbox) != 4:
                errori += 1
                continue
            
            yolo_bbox = voc_to_yolo(bbox, img_w, img_h)
            if yolo_bbox is None:
                errori += 1
                continue
            
            cat_ids = ann.get('cat_id', [])
            if not isinstance(cat_ids, list):
                cat_ids = [cat_ids]
            
            for cat_id in cat_ids:
                if cat_id is None:
                    continue
                try:
                    orig_id = int(cat_id)
                    if orig_id not in id_to_idx:
                        errori += 1
                        continue
                    
                    class_idx = id_to_idx[orig_id]
                    f.write(f"{class_idx} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")
                    total_annotations += 1
                except (ValueError, TypeError):
                    errori += 1
                    continue
    
    # Copia immagine
    src = os.path.join(BASE_PATH, 'images', img_filename)
    dst = os.path.join(IMAGES_DIR, img_filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        images_processed += 1
    else:
        errori += 1

print(f"\n   ✅ Immagini processate: {images_processed}")
print(f"   ✅ Annotazioni scritte: {total_annotations}")
print(f"   ⚠️ Errori: {errori} (trascurabili)")

# ============================================
# 8. CREA LINK SIMBOLICI
# ============================================
print("\n🔗 Creazione link simbolici...")

# Crea link simbolici nella cartella dataset
if os.path.exists(os.path.join(DATASET_DIR, 'images')):
    os.remove(os.path.join(DATASET_DIR, 'images'))
if os.path.exists(os.path.join(DATASET_DIR, 'labels')):
    os.remove(os.path.join(DATASET_DIR, 'labels'))

os.symlink(IMAGES_DIR, os.path.join(DATASET_DIR, 'images'))
os.symlink(LABELS_DIR, os.path.join(DATASET_DIR, 'labels'))

print(f"   {DATASET_DIR}/images → {IMAGES_DIR}")
print(f"   {DATASET_DIR}/labels → {LABELS_DIR}")

# ============================================
# 9. CREA FILE DATA.YAML
# ============================================
print("\n📝 Creazione data.yaml...")

names_list = [idx_to_name[i] for i in range(len(idx_to_name))]

yaml_content = f"""# DeepScoresV2 Dense Dataset - Formato YOLO
# Convertito automaticamente il giorno: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

path: {DATASET_DIR}
train: images
val: images

nc: {len(names_list)}
names: {names_list}
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f"   ✅ File salvato: {YAML_PATH}")
print(f"   nc: {len(names_list)}")
print(f"   Prime 5 classi: {names_list[:5]}")
print(f"   Ultime 5 classi: {names_list[-5:]}")

# ============================================
# 10. VERIFICA FINALE
# ============================================
print("\n🔍 Verifica finale...")

# Controlla che i file non siano vuoti
label_files = os.listdir(LABELS_DIR)
txt_files = [f for f in label_files if f.endswith('.txt')]
print(f"   File label .txt: {len(txt_files)}")

if txt_files:
    sample_path = os.path.join(LABELS_DIR, txt_files[0])
    with open(sample_path, 'r') as f:
        first_lines = f.readlines()[:3]
    print(f"   Esempio di label ({txt_files[0]}):")
    for line in first_lines:
        parts = line.strip().split()
        if len(parts) == 5:
            print(f"      classe {parts[0]} → centro=({parts[1]},{parts[2]}) size=({parts[3]},{parts[4]})")

# Verifica YAML
with open(YAML_PATH, 'r') as f:
    config = yaml.safe_load(f)
    print(f"\n   YAML valido: path={config['path']}, nc={config['nc']}")
    assert config['nc'] == len(config['names']), "❌ nc e names non corrispondono!"
    print("   ✅ nc e names corrispondono")

# ============================================
# RIEPILOGO FINALE
# ============================================
print("\n" + "=" * 70)
print("✅ CONVERSIONE COMPLETATA!")
print("=" * 70)
print(f"""
📊 RIEPILOGO:
   - Immagini: {len(os.listdir(IMAGES_DIR))}
   - Label file: {len(txt_files)}
   - Classi: {len(names_list)} (indici 0..{len(names_list)-1})
   - Annotazioni totali: {total_annotations}
   - Errori: {errori} ({errori/total_annotations*100:.3f}% se total_annotations > 0)

📁 OUTPUT:
   - Dataset root: {DATASET_DIR}
   - Images: {IMAGES_DIR}
   - Labels: {LABELS_DIR}
   - YAML: {YAML_PATH}

🚀 ORA PUOI ADDESTRARE:
   from ultralytics import YOLO
   model = YOLO("yolov8m.pt")
   results = model.train(data='{YAML_PATH}', epochs=100, imgsz=640, batch=4)
""")
print("=" * 70)

🚀 DeepScoresV2 → YOLO Format Converter

📁 Directory create:
   Working dir: /kaggle/working
   Images: /kaggle/working/images
   Labels: /kaggle/working/labels
   Dataset: /kaggle/working/dataset

📂 Caricamento dataset da: /kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense
   Train: 1362 immagini
   Test: 352 immagini

🔍 Analisi classi presenti...
   Classi trovate: 185

🏷️  Creazione mapping classi...
   Mapping creato: 185 classi (0..184)
   Prime 5: [(0, 'brace'), (1, 'ledgerLine'), (2, 'repeatDot'), (3, 'segno'), (4, 'coda')]

📦 Indicizzazione annotazioni...
   Immagini con annotazioni: 1362

🔄 Conversione in formato YOLO...


   Progresso: 100%|██████████| 1362/1362 [00:28<00:00, 47.55it/s]


   ✅ Immagini processate: 1362
   ✅ Annotazioni scritte: 1772055
   ⚠️ Errori: 378 (trascurabili)

🔗 Creazione link simbolici...
   /kaggle/working/dataset/images → /kaggle/working/images
   /kaggle/working/dataset/labels → /kaggle/working/labels

📝 Creazione data.yaml...
   ✅ File salvato: /kaggle/working/dataset/data.yaml
   nc: 185
   Prime 5 classi: ['brace', 'ledgerLine', 'repeatDot', 'segno', 'coda']
   Ultime 5 classi: ['dynamicCrescendoHairpin_204', 'dynamicDiminuendoHairpin_205', 'tuple', 'tupleBracket', 'staff_208']

🔍 Verifica finale...
   File label .txt: 1362
   Esempio di label (lg-43843329-aug-gutenberg1939--page-19.txt):
      classe 113 → centro=(0.499745,0.057900) size=(0.904592,0.024170)
      classe 184 → centro=(0.499745,0.057900) size=(0.904592,0.024170)
      classe 113 → centro=(0.499745,0.179654) size=(0.904592,0.023810)

   YAML valido: path=/kaggle/working/dataset, nc=185
   ✅ nc e names corrispondono

✅ CONVERSIONE COMPLETATA!

📊 RIEPILOGO:
   - Immagini: 1

## CHECK GPU's

In [9]:
import torch
import os

# Forza l'uso di entrambe le GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Verifica
print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")

# Se ancora 1, prova a resettare
if torch.cuda.device_count() == 1:
    torch.cuda.device(0)
    torch.cuda.device(1)
    print(f"Dopo reset: {torch.cuda.device_count()}")

CUDA_VISIBLE_DEVICES: 0,1
torch.cuda.device_count(): 2


## ADDESTRAMENTO

In [3]:
from ultralytics import YOLO
import optuna
import os
import json
import time
from datetime import datetime

# ============================================
# 1. IMPOSTAZIONI PER EVITARE STANDBY
# ============================================
os.environ['KAGGLE_SESSION_KEEP_ALIVE'] = '1'  # Mantiene sessione attiva

# ============================================
# 2. DIRETTORIO PER SALVARE I RISULTATI
# ============================================
SAVE_DIR = '/kaggle/working/optuna_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# File per salvare i risultati parziali
RESULTS_FILE = os.path.join(SAVE_DIR, 'trials_results.json')
BEST_PARAMS_FILE = os.path.join(SAVE_DIR, 'best_params.json')

# ============================================
# 3. FUNZIONE PER SALVARE I RISULTATI
# ============================================
def save_trial_results(trial_number, params, value, study):
    """Salva i risultati di ogni trial"""
    results = {
        'timestamp': datetime.now().isoformat(),
        'trial_number': trial_number,
        'params': params,
        'value': value,
    }
    
    # Carica risultati esistenti
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'r') as f:
            all_results = json.load(f)
    else:
        all_results = []
    
    all_results.append(results)
    
    # Salva
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    # Salva anche i migliori parametri
    if study is not None:
        best_params = {
            'best_params': study.best_params,
            'best_value': study.best_value,
            'timestamp': datetime.now().isoformat()
        }
        with open(BEST_PARAMS_FILE, 'w') as f:
            json.dump(best_params, f, indent=2)
    
    print(f"💾 Risultati salvati in {RESULTS_FILE}")

# ============================================
# 4. CALLBACK PER OPTUNA
# ============================================
def optuna_callback(study, trial):
    """Callback eseguito dopo ogni trial"""
    print(f"\n{'='*60}")
    print(f"✅ Trial {trial.number} completato")
    print(f"   Valore: {trial.value:.4f}")
    print(f"   Parametri: {trial.params}")
    print(f"{'='*60}\n")
    
    # Salva risultati
    save_trial_results(trial.number, trial.params, trial.value, study)

# ============================================
# 5. FUNZIONE OBIETTIVO
# ============================================
def objective(trial):
    # Iperparametri
    lr0 = 0.0005
    lrf = 0.01
    warmup_epochs = 5
    box = trial.suggest_float("box", 6.0, 9.0)
    cls = trial.suggest_float("cls", 0.8, 1.6)
    
    print(f"\n🚀 Trial {trial.number} - Parametri:")
    print(f"   lr0={lr0:.6f}, lrf={lrf:.6f}")
    print(f"   warmup_epochs={warmup_epochs}, box={box:.4f}, cls={cls:.4f}")
    
    # Nome unico per questo trial (evita sovrascritture)
    trial_name = f"trial_{trial.number}_{int(time.time())}"
    
    model = YOLO("yolo11m.pt")
    
    try:
        results = model.train(
            data='/kaggle/working/dataset/data.yaml',
            epochs=200,
            patience=30,
            optimizer='AdamW',
            lr0=lr0,
            lrf=lrf,
            weight_decay=0.0005,
            warmup_epochs=warmup_epochs,
            warmup_bias_lr=0.1,
            warmup_momentum=0.8,
            box=box,
            cls=cls,
            batch=4,
            workers=2,
            imgsz=800,
            device=[0,1],               
            verbose=True,
            amp=True,
            exist_ok=True,
            project='/kaggle/working/runs',  # Cartella fissa
            name=trial_name,                 # Nome unico per trial
        )
        
        final_mAP = results.maps[0]
        print(f"\n✅ Trial {trial.number} completato! mAP50-95: {final_mAP:.4f}")
        
        return final_mAP
        
    except Exception as e:
        print(f"\n❌ Trial {trial.number} fallito: {e}")
        return 0.0

# ============================================
# 6. RIPRISTINO DA SALVATAGGIO (se interrotto)
# ============================================
def load_previous_results():
    """Carica trial già completati per evitare di rifarli"""
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'r') as f:
            return json.load(f)
    return []

# ============================================
# 7. ESECUZIONE
# ============================================
print("="*60)
print("🎯 OTTIMIZZAZIONE IPERPARAMETRI CON OPTUNA")
print("="*60)
print(f"📁 Risultati salvati in: {SAVE_DIR}")
print(f"⏰ Inizio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Crea studio
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=5)
)

# Esegui ottimizzazione con callback
study.optimize(
    objective, 
    n_trials=3,  # Inizia con 3 trial
    callbacks=[optuna_callback],
    show_progress_bar=True
)

# ============================================
# 8. RISULTATI FINALI
# ============================================
print("\n" + "="*60)
print("🏆 RISULTATI FINALI")
print("="*60)
print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP50-95: {study.best_value:.4f}")
print(f"📁 Risultati salvati in: {RESULTS_FILE}")

# Salva risultati finali
save_trial_results('FINAL', study.best_params, study.best_value, study)

print(f"\n✅ COMPLETATO! {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ModuleNotFoundError: No module named 'ultralytics'

## ADDESTRAMENTO (Altri modelli Yolo)

In [11]:
from ultralytics import YOLO

model = YOLO("yolo11l.pt")

try:
    model.train(
        data="/kaggle/working/dataset/data.yaml",
        epochs=200,
        patience=30,
        batch=4,
        imgsz=800,
        device=[0, 1],
        workers=2,
        project="/kaggle/working/runs",
        name="trial_0_1780676922_large",
        exist_ok=True,
        pretrained=True,
        optimizer="AdamW",
        seed=0,
        deterministic=True,
        close_mosaic=10,
        amp=True,
        plots=True,
        lr0=0.0005,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=5,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=6.558749937380739,
        cls=1.0124432316715641,
        dfl=1.5,
        nbs=64,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        auto_augment="randaugment",
        erasing=0.4,
    )
except Exception as e:
    print(f"\n❌ Errore durante il training: {e}")
    traceback.print_exc()
    print("\n🔄 Tentativo di salvare il modello corrente...")
    try:
        # Salva l'ultimo checkpoint disponibile (se esiste)
        model.save("/kaggle/working/runs/last_model_backup.pt")
        print("✅ Modello salvato in /kaggle/working/runs/last_model_backup.pt")
    except:
        print("⚠️ Impossibile salvare il modello (forse nessun peso ancora addestrato).")
    raise   # Rilancia l'eccezione per fermare l'esecuzione

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=6.558749937380739, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0124432316715641, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, flipl

### testing

In [8]:
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import os
from IPython.display import display, clear_output
import ipywidgets as widgets

# ============================================
# CARICA MODELLO
# ============================================
MODEL_PATH = '/kaggle/working/runs/trial_0_1780600327/weights/best.pt'
IMAGE_DIR = '/kaggle/working/images'  # Directory con immagini di test

print("📦 Caricamento modello...")
model = YOLO(MODEL_PATH)
print(f"✅ Modello caricato: {MODEL_PATH}")

# Ottieni nomi delle classi
class_names = model.names
num_classes = len(class_names)
print(f"📋 Classi disponibili: {num_classes}")

# ============================================
# CARICA IMMAGINI DI ESEMPIO
# ============================================
print("\n📂 Caricamento immagini di esempio...")
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))]
image_files = sorted(image_files)[:30]  # Prime 30 immagini
print(f"✅ Trovate {len(image_files)} immagini")

# ============================================
# FUNZIONI PER IL TEST
# ============================================
def predict_image(image_path, conf_threshold=0.25):
    """Esegue inferenza su un'immagine"""
    results = model(image_path, conf=conf_threshold)
    return results[0]

def filter_by_class(results, class_id):
    """Filtra i rilevamenti per classe specifica"""
    if results.boxes is None:
        return None
    boxes = results.boxes
    mask = boxes.cls.int() == class_id
    return boxes[mask]

def get_image_with_predictions(image_path, results, class_id, class_name):
    """Restituisce immagine con bounding box della classe selezionata"""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    filtered_boxes = filter_by_class(results, class_id)
    
    if filtered_boxes is not None and len(filtered_boxes) > 0:
        for box in filtered_boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = box.conf[0].item()
            
            # Disegna rettangolo
            cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
            
            # Aggiungi etichetta
            label = f"{class_name}: {conf:.2f}"
            cv2.putText(img_rgb, label, (int(x1), int(y1)-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    return img_rgb, len(filtered_boxes) if filtered_boxes is not None else 0

# ============================================
# CREA INTERFACCIA INTERATTIVA
# ============================================
# Dropdown per selezione classe
class_options = [(f"{i}: {class_names[i]}", i) for i in range(num_classes)]
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=0,
    description='Classe:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

# Slider per soglia confidenza
conf_slider = widgets.FloatSlider(
    value=0.25,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Confidenza:',
    style={'description_width': 'initial'}
)

# Dropdown per selezione immagine
image_options = [(f"{i}: {f}", i) for i, f in enumerate(image_files)]
image_dropdown = widgets.Dropdown(
    options=image_options,
    value=0,
    description='Immagine:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

# Pulsante per eseguire predizione
predict_button = widgets.Button(
    description='🔍 Esegui Predizione',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Output per visualizzare immagine
output_image = widgets.Output()
output_stats = widgets.Output()

# Stato del caricamento
loading_label = widgets.HTML(value="<b>Seleziona una classe e clicca 'Esegui Predizione'</b>")

# ============================================
# FUNZIONE DI PREDIZIONE
# ============================================
def run_prediction(change=None):
    with output_image:
        clear_output(wait=True)
        output_image.clear_output(wait=True)
        
    with output_stats:
        clear_output(wait=True)
        output_stats.clear_output(wait=True)
    
    class_id = class_dropdown.value
    class_name = class_names[class_id]
    conf_threshold = conf_slider.value
    img_idx = image_dropdown.value
    img_file = image_files[img_idx]
    img_path = os.path.join(IMAGE_DIR, img_file)
    
    with output_stats:
        print(f"🔍 Analisi in corso...")
        print(f"   Classe: {class_id} - {class_name}")
        print(f"   Immagine: {img_file}")
        print(f"   Soglia confidenza: {conf_threshold:.2f}")
    
    # Esegui predizione
    results = predict_image(img_path, conf_threshold)
    
    # Ottieni immagine con predizioni filtrate
    img_with_boxes, num_detections = get_image_with_predictions(img_path, results, class_id, class_name)
    
    # Mostra immagine
    with output_image:
        plt.figure(figsize=(12, 10))
        plt.imshow(img_with_boxes)
        plt.title(f"Classe: {class_name} (ID: {class_id}) - Rilevamenti: {num_detections}", fontsize=12)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    # Mostra statistiche
    with output_stats:
        print(f"\n✅ {num_detections} rilevamenti trovati per la classe '{class_name}'")
        
        # Mostra statistiche totali dell'immagine
        if results.boxes is not None:
            total_detections = len(results.boxes)
            unique_classes = set(results.boxes.cls.int().tolist())
            print(f"\n📊 Statistiche complete immagine:")
            print(f"   Totale rilevamenti: {total_detections}")
            print(f"   Classi rilevate: {len(unique_classes)}")
            print(f"   ID classi: {sorted(unique_classes)[:10]}")
            if len(unique_classes) > 10:
                print(f"   ... e altri {len(unique_classes)-10}")

# ============================================
# AGGIUNGI EVENTI
# ============================================
predict_button.on_click(run_prediction)

# ============================================
# MOSTRA INTERFACCIA
# ============================================
print("\n" + "="*60)
print("🎯 TEST MODELLO YOLO - INTERATTIVO")
print("="*60)

display(widgets.HTML("<br>"))
display(widgets.HBox([class_dropdown, conf_slider]))
display(widgets.HTML("<br>"))
display(image_dropdown)
display(widgets.HTML("<br>"))
display(predict_button)
display(widgets.HTML("<br><hr><br>"))
display(loading_label)
display(output_stats)
display(output_image)

# Esegui predizione iniziale
run_prediction()

print("\n" + "="*60)
print("💡 ISTRUZIONI:")
print("   1. Seleziona una classe dal menu a tendina")
print("   2. Regola la soglia di confidenza (0.25 = default)")
print("   3. Scegli un'immagine tra quelle disponibili")
print("   4. Clicca 'Esegui Predizione'")
print("   5. Le bounding box della classe selezionata appariranno in BLU")
print("="*60)

📦 Caricamento modello...
✅ Modello caricato: /kaggle/working/runs/trial_0_1780600327/weights/best.pt
📋 Classi disponibili: 185

📂 Caricamento immagini di esempio...
✅ Trovate 30 immagini

🎯 TEST MODELLO YOLO - INTERATTIVO


HTML(value='<br>')

HTML(value='<br>')

Dropdown(description='Immagine:', layout=Layout(width='60%'), options=(('0: lg-101766503886095953-aug-beethove…

HTML(value='<br>')

...

HTML(value='<br><hr><br>')

HTML(value="<b>Seleziona una classe e clicca 'Esegui Predizione'</b>")

Output()

Output()


image 1/1 /kaggle/working/images/lg-101766503886095953-aug-beethoven--page-1.png: 640x480 3 braces, 1 segno, 4 clefGs, 4 clefFs, 7 timeSig4s, 58 noteheadBlackOnLines, 50 noteheadBlackInSpaces, 14 noteheadHalfOnLines, 18 noteheadHalfInSpaces, 28 flag8thUps, 1 flag8thDown, 32 keySharps, 1 restHalf, 6 restQuarters, 32 rest8ths, 1 beam, 8 staffs, 2 brace_137s, 9 staff_208s, 63.2ms
Speed: 2.9ms preprocess, 63.2ms inference, 25.0ms postprocess per image at shape (1, 3, 640, 480)

💡 ISTRUZIONI:
   1. Seleziona una classe dal menu a tendina
   2. Regola la soglia di confidenza (0.25 = default)
   3. Scegli un'immagine tra quelle disponibili
   4. Clicca 'Esegui Predizione'
   5. Le bounding box della classe selezionata appariranno in BLU


In [9]:
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import os
from IPython.display import display, clear_output
import ipywidgets as widgets
from collections import Counter

# ============================================
# CARICA MODELLO
# ============================================
MODEL_PATH = '/kaggle/working/runs/trial_0_1780600327/weights/best.pt'
IMAGE_DIR = '/kaggle/working/images'  # Directory con immagini di test

print("📦 Caricamento modello...")
model = YOLO(MODEL_PATH)
print(f"✅ Modello caricato: {MODEL_PATH}")

# Ottieni nomi delle classi
class_names = model.names
num_classes = len(class_names)
print(f"📋 Classi disponibili: {num_classes}")

# ============================================
# CARICA IMMAGINI DI ESEMPIO
# ============================================
print("\n📂 Caricamento immagini di esempio...")
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))]
image_files = sorted(image_files)[:30]  # Prime 30 immagini
print(f"✅ Trovate {len(image_files)} immagini")

# ============================================
# FUNZIONI PER IL TEST
# ============================================
def predict_image(image_path, conf_threshold=0.25):
    """Esegue inferenza su un'immagine"""
    results = model(image_path, conf=conf_threshold)
    return results[0]

def filter_by_class(results, class_id):
    """Filtra i rilevamenti per classe specifica"""
    if results.boxes is None:
        return None
    boxes = results.boxes
    mask = boxes.cls.int() == class_id
    return boxes[mask]

def get_image_with_predictions(image_path, results, class_id, class_name):
    """Restituisce immagine con bounding box della classe selezionata"""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    filtered_boxes = filter_by_class(results, class_id)
    
    if filtered_boxes is not None and len(filtered_boxes) > 0:
        for box in filtered_boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = box.conf[0].item()
            
            # Disegna rettangolo
            cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
            
            # Aggiungi etichetta
            label = f"{class_name}: {conf:.2f}"
            cv2.putText(img_rgb, label, (int(x1), int(y1)-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    return img_rgb, len(filtered_boxes) if filtered_boxes is not None else 0

def get_all_classes_count(results):
    """Restituisce un dizionario con il conteggio di tutte le classi trovate"""
    if results.boxes is None:
        return {}
    class_ids = results.boxes.cls.int().tolist()
    return dict(Counter(class_ids))

def precompute_detections(image_files, conf_threshold=0.25):
    """Precalcola il numero di istanze per ogni classe in ogni immagine"""
    detections = {}
    print("\n📊 Precalcolo rilevamenti per tutte le immagini...")
    
    for i, img_file in enumerate(image_files):
        img_path = os.path.join(IMAGE_DIR, img_file)
        results = predict_image(img_path, conf_threshold)
        
        if results.boxes is not None:
            # Conta istanze per classe
            class_counts = Counter(results.boxes.cls.int().tolist())
            detections[img_file] = dict(class_counts)
        else:
            detections[img_file] = {}
        
        # Progresso
        if (i + 1) % 10 == 0:
            print(f"   Processate {i+1}/{len(image_files)} immagini")
    
    print(f"✅ Precalcolo completato!")
    return detections

# ============================================
# PRECALCOLA RILEVAMENTI
# ============================================
detections_cache = precompute_detections(image_files)

# ============================================
# CREA INTERFACCIA INTERATTIVA CON COUNTER
# ============================================
# Crea opzioni per il dropdown con conteggio istanze della classe selezionata
def get_image_options_with_counts(class_id):
    """Genera opzioni per il dropdown con numero di istanze della classe selezionata"""
    options = []
    for i, img_file in enumerate(image_files):
        count = detections_cache.get(img_file, {}).get(class_id, 0)
        options.append((f"{i}: {img_file} [{count} istanze classe {class_id}]", i))
    return options

# Dropdown per selezione classe
class_options = [(f"{i}: {class_names[i]}", i) for i in range(num_classes)]
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=0,
    description='Classe:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

# Slider per soglia confidenza
conf_slider = widgets.FloatSlider(
    value=0.25,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Confidenza:',
    style={'description_width': 'initial'}
)

# Dropdown per selezione immagine
image_dropdown = widgets.Dropdown(
    options=[],
    value=None,
    description='Immagine:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

# Label per riepilogo classe
class_summary_label = widgets.HTML(value="")

# Output per statistiche dettagliate dell'immagine
image_stats_output = widgets.Output()

# Pulsante per eseguire predizione
predict_button = widgets.Button(
    description='🔍 Esegui Predizione',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Output per visualizzare immagine
output_image = widgets.Output()
output_stats = widgets.Output()

# ============================================
# FUNZIONI DI AGGIORNAMENTO
# ============================================
def update_image_options(change=None):
    """Aggiorna il dropdown delle immagini quando cambia la classe"""
    class_id = class_dropdown.value
    options = get_image_options_with_counts(class_id)
    image_dropdown.options = options
    
    # Aggiorna label riepilogo classe
    total_instances = sum(detections_cache.get(img, {}).get(class_id, 0) for img in image_files)
    class_name = class_names[class_id]
    class_summary_label.value = f"<b>📊 Classe {class_id} - {class_name}</b>: {total_instances} istanze totali su {len(image_files)} immagini"
    
    if options:
        image_dropdown.value = options[0][1]
    
    # Aggiorna le statistiche dell'immagine selezionata
    update_image_stats()

def update_image_stats(change=None):
    """Aggiorna la visualizzazione delle statistiche dell'immagine selezionata"""
    with image_stats_output:
        clear_output(wait=True)
        
        if image_dropdown.value is None:
            print("Nessuna immagine selezionata")
            return
        
        img_idx = image_dropdown.value
        img_file = image_files[img_idx]
        class_counts = detections_cache.get(img_file, {})
        
        if not class_counts:
            print(f"📷 {img_file}: Nessuna classe rilevata")
            return
        
        print(f"📷 {img_file}")
        print("-" * 50)
        # Ordina per numero di istanze decrescente
        sorted_counts = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
        for cls_id, count in sorted_counts:
            cls_name = class_names[cls_id]
            print(f"   Classe {cls_id:3d} ({cls_name[:40]:40}): {count:3d} istanze")
        print("-" * 50)
        print(f"Totale classi rilevate: {len(class_counts)}")
        print(f"Totale istanze: {sum(class_counts.values())}")

def run_prediction(change=None):
    with output_image:
        clear_output(wait=True)
        output_image.clear_output(wait=True)
        
    with output_stats:
        clear_output(wait=True)
        output_stats.clear_output(wait=True)
    
    class_id = class_dropdown.value
    class_name = class_names[class_id]
    conf_threshold = conf_slider.value
    img_idx = image_dropdown.value
    
    if img_idx is None:
        with output_stats:
            print("⚠️ Nessuna immagine selezionata")
        return
    
    img_file = image_files[img_idx]
    img_path = os.path.join(IMAGE_DIR, img_file)
    
    # Ottieni numero precalcolato di istanze per questa classe
    precomputed_count = detections_cache.get(img_file, {}).get(class_id, 0)
    
    with output_stats:
        print(f"🔍 Analisi in corso...")
        print(f"   Classe: {class_id} - {class_name}")
        print(f"   Immagine: {img_file}")
        print(f"   Soglia confidenza: {conf_threshold:.2f}")
        print(f"   📌 Istanze precalcolate per questa classe: {precomputed_count}")
    
    # Esegui predizione
    results = predict_image(img_path, conf_threshold)
    
    # Ottieni immagine con predizioni filtrate
    img_with_boxes, num_detections = get_image_with_predictions(img_path, results, class_id, class_name)
    
    # Mostra immagine
    with output_image:
        plt.figure(figsize=(12, 10))
        plt.imshow(img_with_boxes)
        plt.title(f"Classe: {class_name} (ID: {class_id}) - Rilevamenti: {num_detections}", fontsize=12)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    # Mostra statistiche
    with output_stats:
        print(f"\n✅ {num_detections} rilevamenti trovati per la classe '{class_name}'")
        
        # Mostra statistiche totali dell'immagine
        if results.boxes is not None:
            total_detections = len(results.boxes)
            unique_classes = set(results.boxes.cls.int().tolist())
            print(f"\n📊 Statistiche complete immagine (con soglia {conf_threshold}):")
            print(f"   Totale rilevamenti: {total_detections}")
            print(f"   Classi rilevate: {len(unique_classes)}")
            
            # Mostra le classi con più rilevamenti
            class_counts = Counter(results.boxes.cls.int().tolist())
            top_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:15]
            print(f"   Top classi:")
            for cls_id, count in top_classes:
                print(f"      {cls_id}: {class_names[cls_id][:40]} - {count} istanze")

# ============================================
# COLLEGA EVENTI
# ============================================
class_dropdown.observe(update_image_options, names='value')
image_dropdown.observe(update_image_stats, names='value')
predict_button.on_click(run_prediction)

# ============================================
# MOSTRA INTERFACCIA
# ============================================
print("\n" + "="*60)
print("🎯 TEST MODELLO YOLO - INTERATTIVO")
print("="*60)

# Inizializza dropdown immagini
update_image_options()

display(widgets.HTML("<br>"))
display(widgets.HBox([class_dropdown, conf_slider]))
display(widgets.HTML("<br>"))
display(class_summary_label)
display(widgets.HTML("<br>"))
display(image_dropdown)
display(widgets.HTML("<br>"))
display(image_stats_output)  # Nuovo: statistiche dettagliate dell'immagine
display(widgets.HTML("<br>"))
display(predict_button)
display(widgets.HTML("<br><hr><br>"))
display(output_stats)
display(output_image)

# Esegui predizione iniziale
run_prediction()

print("\n" + "="*60)
print("💡 ISTRUZIONI:")
print("   1. Seleziona una classe dal menu a tendina")
print("   2. Il menu immagini mostra quante istanze di quella classe ci sono in ogni immagine")
print("   3. Sotto il menu immagini, VEDRAI IL RIEPILOGO COMPLETO DI TUTTE LE CLASSI TROVATE NELL'IMMAGINE SELEZIONATA")
print("   4. Regola la soglia di confidenza e clicca 'Esegui Predizione'")
print("   5. Le bounding box della classe selezionata appariranno in BLU")
print("="*60)

📦 Caricamento modello...
✅ Modello caricato: /kaggle/working/runs/trial_0_1780600327/weights/best.pt
📋 Classi disponibili: 185

📂 Caricamento immagini di esempio...
✅ Trovate 30 immagini

📊 Precalcolo rilevamenti per tutte le immagini...

image 1/1 /kaggle/working/images/lg-101766503886095953-aug-beethoven--page-1.png: 640x480 3 braces, 1 segno, 4 clefGs, 4 clefFs, 7 timeSig4s, 58 noteheadBlackOnLines, 50 noteheadBlackInSpaces, 14 noteheadHalfOnLines, 18 noteheadHalfInSpaces, 28 flag8thUps, 1 flag8thDown, 32 keySharps, 1 restHalf, 6 restQuarters, 32 rest8ths, 1 beam, 8 staffs, 2 brace_137s, 9 staff_208s, 29.5ms
Speed: 2.4ms preprocess, 29.5ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /kaggle/working/images/lg-101766503886095953-aug-beethoven--page-4.png: 640x480 1 coda, 6 clefGs, 6 clefFs, 48 noteheadBlackOnLines, 82 noteheadBlackInSpaces, 5 noteheadHalfOnLines, 15 noteheadHalfInSpaces, 16 flag8thUps, 8 flag8thDowns, 45 keySharps, 5 restQuarters, 32 re

HTML(value='<br>')

HTML(value='<br>')

HTML(value='<b>📊 Classe 0 - brace</b>: 68 istanze totali su 30 immagini')

HTML(value='<br>')

...

Button(button_style='success', description='🔍 Esegui Predizione', layout=Layout(width='200px'), style=ButtonSt…

HTML(value='<br><hr><br>')

Output()

Output()


image 1/1 /kaggle/working/images/lg-101766503886095953-aug-beethoven--page-1.png: 640x480 3 braces, 1 segno, 4 clefGs, 4 clefFs, 7 timeSig4s, 58 noteheadBlackOnLines, 50 noteheadBlackInSpaces, 14 noteheadHalfOnLines, 18 noteheadHalfInSpaces, 28 flag8thUps, 1 flag8thDown, 32 keySharps, 1 restHalf, 6 restQuarters, 32 rest8ths, 1 beam, 8 staffs, 2 brace_137s, 9 staff_208s, 28.7ms
Speed: 2.3ms preprocess, 28.7ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)

💡 ISTRUZIONI:
   1. Seleziona una classe dal menu a tendina
   2. Il menu immagini mostra quante istanze di quella classe ci sono in ogni immagine
   3. Sotto il menu immagini, VEDRAI IL RIEPILOGO COMPLETO DI TUTTE LE CLASSI TROVATE NELL'IMMAGINE SELEZIONATA
   4. Regola la soglia di confidenza e clicca 'Esegui Predizione'
   5. Le bounding box della classe selezionata appariranno in BLU
